In [ ]:
from train import train

# select the backbone type. Possible variants are resnet18, resnet50, resnet101, and droneranger
model_variant = "resnet18" #"droneranger"

# NOTE: we tested several variants not included in the paper as they produced worse results.
# Different prediction heads are selected by the alt_head number, where use_alt_head=6 is the version used in the final paper.
# We also tested log distance (set by is_log), and adding x and y to the bounding box features in case of lens distortion
# (set by xy_features), and a dual-backbone (set by crop_only), but none offered improvements.
# alt_head == 0: default drone ranger (DR) prediction head        
# alt_head == 1: alternate 4-layer prediction head
# alt_head == 2: 3-layer prediction head
# alt_head == 3: 4-layer prediction head
# alt_head == 4: DR head + layer norm before head
# alt_head == 5: DR head + layer norm + bbox gate
# alt_head == 6: DR head + bbox gate (OURS)

model, metrics = train(
    data_root="/mnt/active_storage/Knut/LRDD_v3",
    metadata_dir="/mnt/active_storage/Knut/LRDD_v3/metadata",
    backbone=model_variant,
    crop_only=True, # single backbone drone crop for training when true, otherwise uses double backbone crop + full image
    use_huber=True, # use huber loss instead of MSE loss
    use_droneranger=False, # only set this to true when using dronerange backbone
    use_alt_head=6, # uses alternate prediction head, otherwise defaults to dronerange prediction head
    epochs=50,
    batch_size=32,
    device="cuda:1",
    lr=1e-3,
    patience=50,
    checkpoint_path=model_variant+"_huber_220_alt6_xy",
    metrics_path=model_variant+"_huber_220_alt6_xy_training_metrics.json",
    bbox_feature_dim=6,
    num_workers=32,
    crop_size=220,
    is_log=False,
    xy_features=True
)

#checkpoint_weights="resnet50_huber_latest.pth"


In [ ]:
%matplotlib inline
import json
import matplotlib.pyplot as plt

with open("resnet18_huber_220_alt6_xy_training_metrics.json") as f:
    metrics = json.load(f)

plt.plot(metrics["epoch"], metrics["train_loss"], label="Train Loss")
plt.plot(metrics["epoch"], metrics["val_loss"], label="Val Loss")
plt.plot(metrics["epoch"], metrics["val_mae"], label="Val MAE")
plt.plot(metrics["epoch"], metrics["val_rmse"], label="Val RMSE")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from evaluate import evaluate_model

model_variant = "resnet18" #"droneranger"

evaluate_model(
    data_root="/mnt/active_storage/Knut/LRDD_v3",
    metadata_dir="/mnt/active_storage/Knut/LRDD_v3/metadata",
    checkpoint_path=model_variant+"_huber_220_alt6_xy_best.pth",
    backbone=model_variant,
    crop_only=True,
    use_droneranger=False,
    use_alt_head=6, # 
    batch_size=32,
    device="cuda:1",
    save_predictions=False,
    bbox_feature_dim=6,
    num_workers=32,
    max_dist=10000, # set max distance in ft to include for testing. Use 10000 to include all data. I've also been running for 400, 300, and 200ft
    crop_size=220,
    is_log=False,
    xy_features=True
)